# IAN 620 — Module 9: Agentic RAG Lab

This notebook builds Agentic RAG in two stages:

1. a plain-Python controller so every decision is visible,
2. the same workflow with LangGraph.

The running example answers support questions using policy documents and order data.


## 1. Environment

Python 3.11 is a good baseline for this notebook.


In [ ]:
%pip install -q sentence-transformers==6.1.0 langgraph==1.2.16


## 2. Imports


In [ ]:
from datetime import date
from typing import TypedDict

import numpy as np
from sentence_transformers import SentenceTransformer


## 3. Small support knowledge base

The policy documents are intentionally small so the complete workflow stays easy to inspect.


In [ ]:
POLICIES = [
    {
        "source_id": "returns",
        "title": "Standard Return Policy",
        "text": "Standard returns are accepted within 14 days of delivery."
    },
    {
        "source_id": "holiday",
        "title": "Holiday Return Exception",
        "text": "Purchases made in December may be returned within 30 days of delivery."
    },
    {
        "source_id": "refunds",
        "title": "Refund Processing",
        "text": "Approved refunds are returned to the original payment method."
    },
    {
        "source_id": "warranty",
        "title": "Warranty Policy",
        "text": "Defective products may qualify for warranty service after the normal return window."
    }
]

ORDERS = {
    "1042": {
        "order_id": "1042",
        "purchase_date": "2026-09-22",
        "delivered": "2026-09-29",
        "status": "delivered"
    },
    "1201": {
        "order_id": "1201",
        "purchase_date": "2026-12-10",
        "delivered": "2026-12-15",
        "status": "delivered"
    }
}


## 4. Build a simple semantic retriever

We reuse one embedding model and keep the vectors in memory. In a larger application, this function could call Chroma or another vector database.


In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

policy_texts = [item["text"] for item in POLICIES]
policy_vectors = embedding_model.encode(
    policy_texts,
    normalize_embeddings=True
)

def search_policy(query, top_k=3):
    query_vector = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    scores = policy_vectors @ query_vector
    best_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in best_indices:
        item = POLICIES[index].copy()
        item["score"] = float(scores[index])
        results.append(item)

    return results


In [ ]:
search_policy("return deadline after delivery", top_k=2)


## 5. Structured order lookup

This is a normal deterministic tool. The LLM does not need to invent or estimate order facts.


In [ ]:
def get_order(order_id):
    order = ORDERS.get(str(order_id))

    if order is None:
        return {
            "found": False,
            "order_id": str(order_id)
        }

    return {
        "found": True,
        **order
    }

get_order("1042")


## 6. Keep agent state explicit


In [ ]:
def make_state(question):
    return {
        "question": question,
        "query": question,
        "route": None,
        "documents": [],
        "order": None,
        "order_id": None,
        "steps": 0,
        "status": "running",
        "answer": None,
        "sources": [],
        "trace": []
    }


## 7. Route the question

Start with transparent rules. Later, this function can be replaced with structured LLM output.


In [ ]:
import re

def extract_order_id(text):
    match = re.search(r"\b(?:order\s*)?(\d{4})\b", text, flags=re.IGNORECASE)
    return match.group(1) if match else None

def route_question(question):
    text = question.lower()
    order_id = extract_order_id(question)

    if order_id and any(word in text for word in ["return", "refund", "policy", "eligible"]):
        route = "policy_and_order"
    elif order_id:
        route = "order_lookup"
    elif any(word in text for word in ["return", "refund", "policy", "warranty"]):
        route = "policy_search"
    else:
        route = "answer_directly"

    return {
        "route": route,
        "order_id": order_id
    }

route_question("Can I return order 1042?")


## 8. Grade the evidence

Relevant evidence is not always sufficient evidence. A policy question about a specific order needs both policy text and order facts.


In [ ]:
def grade_evidence(state):
    has_policy = len(state["documents"]) > 0

    if state["route"] == "policy_and_order":
        has_order = bool(state["order"] and state["order"].get("found"))
        sufficient = has_policy and has_order
    elif state["route"] == "policy_search":
        sufficient = has_policy
    elif state["route"] == "order_lookup":
        sufficient = bool(state["order"] and state["order"].get("found"))
    else:
        sufficient = True

    return {
        "relevant": has_policy or state["route"] in {"order_lookup", "answer_directly"},
        "sufficient": sufficient
    }


## 9. Rewrite a weak retrieval query

This notebook keeps the rewrite deterministic so the workflow runs without an API key.


In [ ]:
def rewrite_query(question, documents):
    text = question.lower()

    if "return" in text or "refund" in text:
        return "return eligibility deadline after delivery refund policy"

    if "warranty" in text:
        return "warranty eligibility after return window"

    return question


## 10. Build a grounded answer

For the lab, the answer generator is deterministic. The orchestration logic is the important part. You can later replace this function with a local or hosted LLM.


In [ ]:
def days_since(delivered_date, today="2026-10-19"):
    delivered = date.fromisoformat(delivered_date)
    current = date.fromisoformat(today)
    return (current - delivered).days

def generate_answer(state):
    docs = state["documents"]
    order = state["order"]

    sources = [doc["source_id"] for doc in docs]

    if state["route"] == "policy_and_order":
        if not order or not order.get("found"):
            return {
                "answer": "I could not verify the order details.",
                "sources": sources
            }

        elapsed = days_since(order["delivered"])

        standard = next(
            (doc for doc in docs if doc["source_id"] == "returns"),
            None
        )

        if standard and elapsed > 14:
            answer = (
                f"Order {order['order_id']} was delivered {elapsed} days ago. "
                "The standard return policy allows returns within 14 days of delivery, "
                "so the standard window has passed."
            )
        else:
            answer = (
                f"Order {order['order_id']} was delivered {elapsed} days ago. "
                "The available policy evidence should be reviewed for eligibility."
            )

        return {
            "answer": answer,
            "sources": sources
        }

    if docs:
        return {
            "answer": docs[0]["text"],
            "sources": sources
        }

    return {
        "answer": "I do not have enough verified evidence to answer.",
        "sources": []
    }


## 11. Validate citations


In [ ]:
def validate_sources(result, documents):
    allowed = {
        doc["source_id"]
        for doc in documents
    }

    return all(
        source_id in allowed
        for source_id in result["sources"]
    )


## 12. Plain-Python Agentic RAG controller

The controller is bounded. It records a trace and stops after a small number of steps.


In [ ]:
def run_agent(question, max_steps=3):
    state = make_state(question)

    decision = route_question(question)
    state["route"] = decision["route"]
    state["order_id"] = decision["order_id"]

    state["trace"].append({
        "step": 0,
        "action": "route",
        "route": state["route"]
    })

    for step in range(1, max_steps + 1):
        state["steps"] = step

        if state["route"] in {"policy_search", "policy_and_order"}:
            state["documents"] = search_policy(
                state["query"],
                top_k=3
            )

        if state["route"] in {"order_lookup", "policy_and_order"}:
            state["order"] = get_order(
                state["order_id"]
            )

        grade = grade_evidence(state)

        state["trace"].append({
            "step": step,
            "action": "grade",
            "sufficient": grade["sufficient"],
            "query": state["query"]
        })

        if grade["sufficient"]:
            result = generate_answer(state)

            if validate_sources(result, state["documents"]):
                state["answer"] = result["answer"]
                state["sources"] = result["sources"]
                state["status"] = "answer_ready"
            else:
                state["status"] = "validation_error"

            break

        new_query = rewrite_query(
            state["question"],
            state["documents"]
        )

        if new_query == state["query"]:
            state["status"] = "no_progress"
            break

        state["query"] = new_query
        state["trace"].append({
            "step": step,
            "action": "rewrite",
            "query": new_query
        })

    if state["status"] == "running":
        state["status"] = "max_steps"

    return state


In [ ]:
result = run_agent(
    "Order 1042 was delivered about 20 days ago. Can I still return it?"
)

print("Status:", result["status"])
print("Answer:", result["answer"])
print("Sources:", result["sources"])

result["trace"]


## 13. Compare a policy-only question


In [ ]:
result = run_agent(
    "What is the standard return deadline?"
)

print(result["answer"])
print(result["trace"])


## 14. LangGraph version

The same functions can become graph nodes.


In [ ]:
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict):
    question: str
    query: str
    route: str
    documents: list
    order: dict | None
    order_id: str | None
    steps: int
    status: str
    answer: str | None
    sources: list
    trace: list


## 15. Graph nodes


In [ ]:
def route_node(state: AgentState):
    decision = route_question(state["question"])

    return {
        "route": decision["route"],
        "order_id": decision["order_id"],
        "query": state["question"],
        "trace": state["trace"] + [{
            "action": "route",
            "route": decision["route"]
        }]
    }

def retrieve_node(state: AgentState):
    docs = state["documents"]
    order = state["order"]

    if state["route"] in {"policy_search", "policy_and_order"}:
        docs = search_policy(state["query"], top_k=3)

    if state["route"] in {"order_lookup", "policy_and_order"}:
        order = get_order(state["order_id"])

    return {
        "documents": docs,
        "order": order,
        "steps": state["steps"] + 1
    }

def grade_node(state: AgentState):
    grade = grade_evidence(state)

    return {
        "status": (
            "sufficient"
            if grade["sufficient"]
            else "needs_rewrite"
        ),
        "trace": state["trace"] + [{
            "action": "grade",
            "sufficient": grade["sufficient"]
        }]
    }

def rewrite_node(state: AgentState):
    new_query = rewrite_query(
        state["question"],
        state["documents"]
    )

    return {
        "query": new_query,
        "status": "running",
        "trace": state["trace"] + [{
            "action": "rewrite",
            "query": new_query
        }]
    }

def generate_node(state: AgentState):
    result = generate_answer(state)

    valid = validate_sources(
        result,
        state["documents"]
    )

    return {
        "answer": result["answer"] if valid else None,
        "sources": result["sources"] if valid else [],
        "status": "answer_ready" if valid else "validation_error"
    }


## 16. Conditional transition


In [ ]:
def after_grade(state: AgentState):
    if state["status"] == "sufficient":
        return "generate"

    if state["steps"] >= 3:
        return "stop"

    return "rewrite"


## 17. Build and compile the graph


In [ ]:
graph = StateGraph(AgentState)

graph.add_node("route", route_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("grade", grade_node)
graph.add_node("rewrite", rewrite_node)
graph.add_node("generate", generate_node)

graph.add_edge(START, "route")
graph.add_edge("route", "retrieve")
graph.add_edge("retrieve", "grade")

graph.add_conditional_edges(
    "grade",
    after_grade,
    {
        "generate": "generate",
        "rewrite": "rewrite",
        "stop": END
    }
)

graph.add_edge("rewrite", "retrieve")
graph.add_edge("generate", END)

app = graph.compile()


## 18. Run the graph


In [ ]:
initial_state = {
    "question": "Can I return order 1042?",
    "query": "",
    "route": "",
    "documents": [],
    "order": None,
    "order_id": None,
    "steps": 0,
    "status": "running",
    "answer": None,
    "sources": [],
    "trace": []
}

graph_result = app.invoke(initial_state)

print(graph_result["status"])
print(graph_result["answer"])
graph_result["trace"]


## 19. Agent evaluation cases

The final answer is only one part of evaluation. Check the route and trajectory too.


In [ ]:
TEST_CASES = [
    {
        "question": "What is the return deadline?",
        "expected_route": "policy_search"
    },
    {
        "question": "When was order 1042 delivered?",
        "expected_route": "order_lookup"
    },
    {
        "question": "Can I return order 1042?",
        "expected_route": "policy_and_order"
    }
]

for case in TEST_CASES:
    decision = route_question(
        case["question"]
    )

    print(
        case["question"],
        decision["route"],
        decision["route"] == case["expected_route"]
    )


## 20. Production checklist

Before using an Agentic RAG workflow in a real application, verify:

- allowed tools are explicit,
- tool arguments are validated,
- permissions are enforced outside the LLM,
- retries and maximum steps are bounded,
- retrieved content is treated as untrusted data,
- source provenance is preserved,
- traces do not contain secrets,
- evaluation includes routing, retrieval, trajectory, and answer quality.
